# Week 3 Exercise — Synthetic Data Generator

Generate **synthetic datasets** from a natural-language description: describe the scenario (e.g. "customer support tickets with priority, status, and summary"), choose number of rows and output format (JSON or CSV). Uses **OpenRouter** (and optional **Ollama**) so it runs locally without Colab or HuggingFace.

Set `OPENROUTER_API_KEY` in `.env`. For Ollama: `ollama pull llama3.2` and have Ollama running.

In [ ]:
import os
import json
import re
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    print("Add OPENROUTER_API_KEY to .env")
else:
    print("OpenRouter API key loaded.")

MODELS = {
    "OpenRouter: gpt-4o-mini": ("openai/gpt-4o-mini", "https://openrouter.ai/api/v1", api_key),
    "OpenRouter: claude-3-haiku": ("anthropic/claude-3-haiku", "https://openrouter.ai/api/v1", api_key),
    "Ollama: llama3.2": ("llama3.2", "http://localhost:11434/v1", "ollama"),
}

def get_client(model_key):
    model_id, base_url, key = MODELS.get(model_key, list(MODELS.values())[0])
    return OpenAI(api_key=key or "ollama", base_url=base_url), model_id

In [ ]:
SYSTEM_PROMPT = """You are an expert synthetic data generator. Generate realistic but fake data only. No real names, emails, or PII.
Output ONLY the requested structure: a JSON array of objects, or a CSV with header row. No markdown, no code fences, no explanation.
Keep values diverse and plausible. Match the user's schema exactly."""

In [ ]:
def generate_synthetic_data(scenario: str, num_rows: int, fmt: str, model_key: str) -> tuple:
    if not scenario or not scenario.strip():
        return "Describe the dataset you want (e.g. columns and type of data).", None
    client, model_id = get_client(model_key)
    num_rows = max(1, min(50, int(num_rows)))
    user_msg = f"""Generate exactly {num_rows} synthetic records for this scenario:

{scenario.strip()}

Requirements:
- Output ONLY data, no explanation.
- {"Output as a JSON array of objects, one object per record. Keys should be snake_case." if fmt == "JSON" else "Output as CSV: first line is header, then one row per record. Use comma separator."}
"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg}
    ]
    try:
        r = client.chat.completions.create(model=model_id, messages=messages)
        text = (r.choices[0].message.content or "").strip()
        text = re.sub(r"^```(?:json|csv)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
        if fmt == "JSON":
            json.loads(text)
        out_path = "synthetic_data.json" if fmt == "JSON" else "synthetic_data.csv"
        with open(out_path, "w") as f:
            f.write(text)
        return text, out_path
    except Exception as e:
        return f"Error: {e}", None

In [ ]:
with gr.Blocks(title="Synthetic Data Generator") as demo:
    gr.Markdown("# Week 3 — Synthetic Data Generator")
    scenario = gr.Textbox(
        label="Scenario",
        placeholder="e.g. 10 product reviews with: title, rating (1-5), body, date",
        lines=3
    )
    num_rows = gr.Slider(1, 50, value=5, step=1, label="Number of rows")
    fmt = gr.Radio(["JSON", "CSV"], value="JSON", label="Output format")
    model_dd = gr.Dropdown(choices=list(MODELS.keys()), value=list(MODELS.keys())[0], label="Model")
    btn = gr.Button("Generate")
    out_text = gr.Textbox(label="Output", lines=12)
    out_file = gr.File(label="Download")
    btn.click(
        fn=generate_synthetic_data,
        inputs=[scenario, num_rows, fmt, model_dd],
        outputs=[out_text, out_file]
    )
    gr.Examples(
        [
            ["5 customer support tickets with: id, priority (low/medium/high), status, summary", 5, "JSON", list(MODELS.keys())[0]],
            ["10 employees: name, department, salary, years_experience", 10, "CSV", list(MODELS.keys())[0]],
        ],
        inputs=[scenario, num_rows, fmt, model_dd],
        label="Examples"
    )
demo.launch()